In [1]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

# Load your code.env file
env_file = r"C:\Users\Abhijeet\code.env"

with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

print(f" Loaded from: {env_file}")

ok_openai = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI : {' Loaded' if ok_openai else ' Not found'}")

 Loaded from: C:\Users\Abhijeet\code.env
OpenAI :  Loaded


In [2]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"],
               capture_output=True)

# Load from code.env
env_file = r"C:\Users\Abhijeet\code.env"
with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

# Anthropic not needed — set dummy so scraper skips quietly
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = "not-used"

ok = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI key : {' Ready' if ok else ' Not found — check code.env'}")
if ok:
    print(" Ready to run scraper")

OpenAI key :  Ready
 Ready to run scraper


In [3]:
import asyncio
import re
import sys
import json
import urllib.request
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime


# ── SENTENCE-SAFE TRUNCATION 

def clean_to_sentences(text: str, max_sentences: int = 2) -> str:
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    good = [
        s.strip() for s in sentences
        if len(s.strip().split()) >= 6 and re.search(r'[.!?]$', s.strip())
    ]
    return " ".join(good[:max_sentences])


# ── API KEY 

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def _get_openai_key() -> str:
    return os.environ.get("OPENAI_API_KEY", "")

def _check_key(key: str, name: str) -> bool:
    if not key or key.startswith("YOUR_") or len(key) < 20:
        print(f"  ⚠ {name} not set — add it to your .env file")
        return False
    return True


# ── HELPERS 

async def highlight_and_click(page, selector_or_locator, description="Action"):
    try:
        target = (
            page.locator(selector_or_locator).first
            if isinstance(selector_or_locator, str)
            else selector_or_locator
        )
        if await target.is_visible(timeout=2000):
            await target.evaluate(
                "el => { el.style.border = '4px solid #00FF00';"
                " el.style.backgroundColor = 'rgba(0,255,0,0.2)'; }"
            )
            await target.click()
            return True
    except Exception:
        pass
    return False


async def handle_cookies_automatically(page):
    for selector in [
        "text=Accept All", "text=Accept",
        "button:has-text('Accept')", "button:has-text('OK')",
        "button:has-text('I Agree')", "button:has-text('Close')",
        "button:has-text('Got it')", "button:has-text('Agree')",
    ]:
        if await highlight_and_click(page, selector, "Cookie Banner"):
            break


async def smart_goto(page, url, timeout=90000):
    try:
        await page.goto(url, wait_until="networkidle", timeout=timeout)
        return True
    except Exception as e:
        print(f"  ⚠ networkidle failed ({e.__class__.__name__}), retrying…")
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout)
        await asyncio.sleep(6)
        try:
            await page.wait_for_function(
                "() => document.body && document.body.innerText.trim().length > 200",
                timeout=15000)
        except Exception:
            pass
        return True
    except Exception as e:
        try:
            await page.goto(url, wait_until="commit", timeout=timeout)
            await asyncio.sleep(8)
            return True
        except Exception:
            pass
        print(f"  ✗ Failed: {e}")
        return False


# ── CITY DETECTION 

CITY_HINTS = {
    "singapore": "Singapore", "buona vista": "Singapore", "orchard": "Singapore",
    "clementi": "Singapore", "jurong": "Singapore", "tampines": "Singapore",
    "ang mo kio": "Singapore", "novena": "Singapore", "toa payoh": "Singapore",
    "bishan": "Singapore", "woodlands": "Singapore", "bedok": "Singapore",
    "punggol": "Singapore", "queenstown": "Singapore", "newton": "Singapore",
    "raffles": "Singapore", ".sg": "Singapore", "edu.sg": "Singapore",
    "org.sg": "Singapore", "com.sg": "Singapore",
    "kuala lumpur": "Kuala Lumpur, Malaysia", "jakarta": "Jakarta, Indonesia",
    "hong kong": "Hong Kong", "dubai": "Dubai, UAE",
    "london": "London, UK", "new york": "New York, US",
}

def detect_city(url: str, body_text: str) -> str:
    for hint, city in CITY_HINTS.items():
        if hint in url.lower():
            return city
    for hint, city in CITY_HINTS.items():
        if hint in body_text[:3000].lower():
            return city
    return "Singapore"


# ── GARBAGE SITE DETECTION 

GARBAGE_SIGNALS = [
    "hugedomains", "domain for sale", "buy this domain", "sedoparking",
    "this domain is for sale", "godaddy", "namecheap parking",
    "page not found", "account suspended", "coming soon",
    "under construction", "website coming soon",
]

def is_garbage_site(name: str, body_text: str) -> bool:
    combined = (name + " " + body_text[:500]).lower()
    return any(sig in combined for sig in GARBAGE_SIGNALS)


# ── JUNK FILTER 

JUNK_PATTERNS = [
    r"the site you are about to visit", r"page you requested no longer exists",
    r"managed independently", r"beginning of dialog window", r"escape will cancel",
    r"enquire now$", r"apply now$", r"watch the video", r"learn more$",
    r"find out more$", r"read more$", r"click here",
    r'"school" means the school', r"student.*means the child",
    r"parents.*guardians.*applying", r"please fill out the form",
    r"fill the registration form", r"step \d+ - complete",
    r"mailing list.*keep you up to date", r"joining our mailing list",
    r"to progress your application we require",
    r"please see below the list of documents",
    r"documents that are mandatory to upload",
    r"our system is limited to uploading", r"scans of multi-page documents",
    r"tourist visa.*online", r"apply for your tourist visa",
    r"cookie", r"privacy policy", r"gdpr",
    r"we will respond to your request",
    r"managing and terminating the employment contract",
    r"personal data unless \(a\) it is provided",
    r"report lost items", r"recover belongings",
]

def is_junk(text: str) -> bool:
    if not text:
        return True
    tl = text.lower().strip()
    return any(re.search(p, tl) for p in JUNK_PATTERNS)


# ── IMAGE HARVESTING 

SKIP_IMG = [
    "logo", "icon", "svg", "1x1", "pixel", "spinner", "placeholder",
    "blank", "transparent", "avatar", "flag", "arrow", "bullet",
    "check", "star", "rating", "rs/facebook", "rs/linkedin", "rs/youtube",
    "loupe", "panier", "menu", "profil", "default_image", "black-beacon",
    "weather/128",
]
EXTS_IMG = [".jpg", ".jpeg", ".png", ".webp"]

async def harvest_images(pg, base_url: str, image_set: set):
    try:
        await pg.evaluate("""async () => {
            await new Promise(resolve => {
                let total = 0;
                const dist = 400;
                const timer = setInterval(() => {
                    window.scrollBy(0, dist);
                    total += dist;
                    if (total >= document.body.scrollHeight) {
                        clearInterval(timer);
                        window.scrollTo(0, 0);
                        resolve();
                    }
                }, 80);
            });
        }""")
        await asyncio.sleep(1)
    except Exception:
        pass

    try:
        for img in await pg.query_selector_all("img"):
            for attr in ["src", "data-src", "data-lazy-src", "data-original",
                         "data-image", "data-bg", "data-lazy", "data-srcset"]:
                c = await img.get_attribute(attr)
                if c:
                    src = c.strip().split(" ")[0].split(",")[0].strip()
                    full = urljoin(base_url, src)
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)
                            and len(full) > 25):
                        image_set.add(full)
            srcset = await img.get_attribute("srcset")
            if srcset:
                parts = [p.strip().split(" ")[0] for p in srcset.split(",") if p.strip()]
                if parts:
                    full = urljoin(base_url, parts[-1])
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)):
                        image_set.add(full)

        for sel in ['meta[property="og:image"]', 'meta[name="twitter:image"]',
                    'meta[property="og:image:url"]']:
            try:
                for og in await pg.evaluate(
                    f"() => Array.from(document.querySelectorAll('{sel}')).map(m=>m.content)"
                ):
                    if og and og.strip() and not any(s in og.lower() for s in SKIP_IMG):
                        image_set.add(og.strip())
            except Exception:
                pass

        bg_urls = await pg.evaluate("""() => {
            const hits = [];
            const sels = [
                '[style*="background-image"]','[class*="hero"]','[class*="banner"]',
                '[class*="slider"]','[class*="slide"]','[class*="bg-image"]',
                '[class*="cover"]','section','header'
            ];
            sels.forEach(sel => {
                try {
                    document.querySelectorAll(sel).forEach(el => {
                        const s = el.style.backgroundImage ||
                                  getComputedStyle(el).backgroundImage || '';
                        const m = s.match(/url\\(["']?([^"')]+)["']?\\)/);
                        if (m && m[1]) hits.push(m[1]);
                    });
                } catch(e) {}
            });
            return [...new Set(hits)];
        }""")
        for bg in bg_urls:
            if bg:
                full = urljoin(base_url, bg.strip())
                if (any(e in full.lower() for e in EXTS_IMG)
                        and not any(s in full.lower() for s in SKIP_IMG)):
                    image_set.add(full)
    except Exception:
        pass


# ── QUOTE SCRAPER 

def _word_count(text):
    return len(text.split())

def scrape_quote_from_soup(soup):
    SKIP_PHRASES = [
        "cookie", "menu", "nav", "©", "privacy", "terms",
        "subscribe", "sign up", "click", "read more", "javascript",
        "khda", "rated", "inspection", "watch the video", "watch video",
        "listen to", "testimonial", "share", "follow us", "contact us",
        "apply now", "find out more", "learn more", "get in touch",
        "download", "register", "book a", "visit us",
        "means the child", "parents/guardians", "dialog window",
        "escape will cancel", "site you are about to visit",
        "no longer exists", "managed independently",
    ]
    selectors = [
        "blockquote", "q",
        "[class*='quote']", "[class*='pullquote']", "[class*='callout']",
        "[class*='hero'] p", "[class*='banner'] p",
        "[class*='mission'] p", "[class*='vision'] p",
        "figcaption",
    ]
    candidates = []
    for sel in selectors:
        for el in soup.select(sel):
            text = re.sub(r'\s+', ' ', el.get_text()).strip()
            text = text.strip('\u201c\u201d\u2018\u2019"\'')
            wc = _word_count(text)
            if wc < 8 or wc > 35:
                continue
            if any(x in text.lower() for x in SKIP_PHRASES):
                continue
            if is_junk(text):
                continue
            if not re.search(r'[.!?]$', text):
                continue
            candidates.append(text)
    if not candidates:
        return ""
    candidates.sort(key=lambda t: _word_count(t))
    return candidates[0]


# ── AI QUOTE 

def generate_ai_quote(school_name, school_type, city, about_text):
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return f"Excellence, integrity, and curiosity — the pillars of {school_name}."
    prompt = (
        f"Write ONE short inspirational quote (1 sentence, 10–20 words) "
        f"that reflects the philosophy of {school_name}, a {school_type} in {city}. "
        f"Context: {about_text[:300]}. "
        f"Sound like the organisation's mission statement. "
        f"Return ONLY the quote text — no quotation marks, no attribution, no explanation."
    )
    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 60, "temperature": 0.7,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = json.loads(resp.read())
            quote = data["choices"][0]["message"]["content"].strip().strip('"\'')
            words = quote.split()
            if len(words) > 25:
                quote = " ".join(words[:25])
            print(f"  ✓ AI quote generated for {school_name}")
            return quote
    except Exception as e:
        print(f"  ⚠ AI quote failed: {e}")
    return f"Excellence, integrity, and curiosity — the pillars of {school_name}."


# ── KNOWN NAMES MAP 

KNOWN_NAMES_SG = {
    "tts.edu.sg":            "Tanglin Trust School",
    "ofs.edu.sg":            "Overseas Family School",
    "sas.edu.sg":            "Singapore American School",
    "cis.edu.sg":            "Canadian International School",
    "sportsschool.edu.sg":   "Singapore Sports School",
    "sswimclub.org.sg":      "Singapore Swimming Club",
    "tanglinclub.org":       "The Tanglin Club",
    "scc.org.sg":            "Singapore Cricket Club",
    "prep-zone.sg":          "Prep Zone Academy",
    "thelearninglab.com.sg": "The Learning Lab",
    "crimsoneducation.org":  "Crimson Education",
    "lasalle.edu.sg":        "LASALLE College of the Arts",
    "sso.org.sg":            "Singapore National Youth Orchestra",
    "brahmcentre.com":       "Brahm Centre",
    "centreformindfulness.sg": "Centre for Mindfulness Singapore",
    "redefinewellness.asia": "ReDefine Wellness",
    "corecollective.sg":     "Core Collective",
    "psychconnect.sg":       "Psych Connect",
    "srmc.edu.sg":           "Singapore Raffles Music College",
    "sota.edu.sg":           "School of the Arts Singapore",
}


# ── SHORT KEYWORDS MAP for WP blocks 

# These replace the long metric headings with short punchy WP keywords
PERF_SHORT = {
    "Coaching Credentials":   "Expert Staff",
    "Student Wellbeing":      "Student Care",
    "Academic Integration":   "Curriculum",
    "Competitive Pathway":    "Achievements",
    "Facilities & Resources": "Facilities",
    "Ongoing Accountability": "Progress",
}


# ── OPENAI FORCE-FILL ALL PERFORMANCE FIELDS 

def openai_force_fill_performance(results: dict) -> dict:
    """
    Force-fills ALL 6 performance fields using OpenAI.
    Any field that is blank, N/A, too short (<12 words), or contains junk
    is sent to OpenAI for a unique, substantive 2-sentence response.
    NEVER overwrites a genuinely good scraped value.
    """
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    name = results.get("Name", "this organisation")
    entity_type = results.get("Type", "Independent School")
    city = results.get("City", "Singapore")
    about = results.get("About", "")
    philosophy = results.get("Philosophy", "")

    # Build context
    ctx_parts = [f"Name: {name}", f"Type: {entity_type}", f"City: {city}"]
    if about:
        ctx_parts.append(f"About: {about}")
    if philosophy:
        ctx_parts.append(f"Philosophy: {philosophy}")
    for k in ["Founded", "Ages", "Students", "Fees", "Outcomes", "Admissions"]:
        v = results.get(k, "")
        if v and v not in ("N/A", "See website", ""):
            ctx_parts.append(f"{k}: {v}")
    context = "\n".join(ctx_parts)

    # All 6 performance metric prompts — specific, distinct angles
    PERF_PROMPTS = {
        "Coaching Credentials": (
            f"2 sentences (≤35 words total) about the qualifications, expertise, or professional "
            f"background of staff/coaches/teachers at {name}, a {entity_type} in {city}. "
            f"Mention certifications, experience level, or teaching approach. Be specific."
        ),
        "Student Wellbeing": (
            f"2 sentences (≤35 words total) about how {name} supports the mental, emotional, "
            f"or physical wellbeing of students/members/clients. Mention pastoral care, "
            f"counselling, community, or support systems. Be specific."
        ),
        "Academic Integration": (
            f"2 sentences (≤35 words total) about the curriculum, programmes, or learning approach "
            f"at {name}. Mention specific subjects, qualifications (IB/IGCSE/A-Level etc), "
            f"or educational philosophy. Be specific."
        ),
        "Competitive Pathway": (
            f"2 sentences (≤35 words total) about the results, achievements, university "
            f"destinations, competition results, or career pathways from {name}. "
            f"Mention tangible outcomes or success metrics. Be specific."
        ),
        "Facilities & Resources": (
            f"2 sentences (≤35 words total) about the physical facilities, campus spaces, "
            f"equipment, technology, or resources available at {name}. "
            f"Mention specific amenities like studios, labs, pools, etc. Be specific."
        ),
        "Ongoing Accountability": (
            f"2 sentences (≤35 words total) about how {name} tracks student/member progress, "
            f"provides feedback, monitors quality, or reports to stakeholders. "
            f"Mention reviews, assessments, parent communication, or quality systems. Be specific."
        ),
    }

    def needs_fill(val: str) -> bool:
        if not val or val.strip() in ("N/A", "", "N/a", "See website"):
            return True
        clean = re.sub(r'\s+', ' ', val).strip()
        if len(clean.split()) < 12:
            return True
        # Junk detection
        junk_phrases = [
            "cookie", "privacy policy", "gdpr", "javascript", "tourist visa",
            "we will respond to your request", "managing and terminating",
            "personal data unless", "report lost items", "login with your",
            "click here", "read more", "learn more", "find out more",
            "from time to time, it may be necessary for employees",
            "the platform evaluates what responsible gambling",
            "mckenzie, founder of the american mental health",
            "businesses train employees in silva method",
        ]
        cl = clean.lower()
        return any(p in cl for p in junk_phrases)

    # Identify which fields need filling
    missing = {
        metric: prompt
        for metric, prompt in PERF_PROMPTS.items()
        if needs_fill(results.get("Performance", {}).get(metric, ""))
    }

    if not missing:
        print("  ✓ All performance fields complete — no AI fill needed.")
        return results

    print(f"  → AI force-filling {len(missing)} performance field(s): {', '.join(missing.keys())}")

    # Build prompt — ask for all missing fields in one call
    field_lines = "\n".join(
        f'  "{k}": {v}' for k, v in missing.items()
    )

    prompt = f"""You are an expert on Singapore educational institutions, wellness centres, arts organisations, and professional clubs.

KNOWN DATA about this institution:
{context}

Generate UNIQUE, FACTUAL, SPECIFIC content for ONLY these fields.
Each field must:
- Be exactly 2 sentences, ≤35 words total
- Describe a DIFFERENT aspect from the other fields  
- Sound professional and informative (not generic)
- Use your knowledge of {name} in {city} to be accurate
- Never repeat facts from another field

Reply ONLY with valid JSON, keys exactly as shown below. No markdown, no extra keys:

{field_lines}

Return format: {{"field name": "2 sentence content here.", ...}}"""

    payload = json.dumps({
        "model": "gpt-4o",
        "max_tokens": 1200,
        "temperature": 0.4,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=40) as resp:
            data = json.loads(resp.read())
            raw = data["choices"][0]["message"]["content"].strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw = re.sub(r"\s*```$", "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            seen_vals = set()
            for metric, val in filled.items():
                if not val or val.strip() in ("N/A", ""):
                    continue
                # Dedup — don't use same content in two fields
                key_short = re.sub(r'\s+', ' ', val.strip().lower())[:100]
                if key_short in seen_vals:
                    continue
                seen_vals.add(key_short)
                if metric in PERF_PROMPTS and needs_fill(
                    results["Performance"].get(metric, "")
                ):
                    results["Performance"][metric] = str(val).strip()
                    filled_keys.append(metric)

            if filled_keys:
                print(f"  ✓ AI filled performance: {', '.join(filled_keys)}")
            else:
                print("  ✓ AI performance: nothing new added.")

    except json.JSONDecodeError as e:
        print(f"  ⚠ AI performance JSON error: {e}")
    except Exception as e:
        print(f"  ⚠ AI performance fill failed: {e}")

    return results


# ── OPENAI MAIN FIELDS GAP-FILL 

def openai_fill_missing_fields(results: dict) -> dict:
    """Fill main fields (Founded, Ages, Students, Ratio, Fees, About, etc.)"""
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    entity_type = results.get("Type", "Independent School")
    is_school = any(w in entity_type.lower()
                    for w in ["school", "academy", "college", "education", "orchestra", "conservat"])
    name = results.get("Name", "this organisation")

    FILL_FIELDS = {
        "Founded":    "Year this organisation was founded (just the 4-digit year). If unknown reply N/A.",
        "Ages":       ("Student age range (e.g. '3–18'). If unknown reply N/A." if is_school
                       else "Age range of members/clients served (e.g. 'Adults', '5–80', 'All ages'). Never reply N/A."),
        "Students":   ("Approximate enrolled students (e.g. '1200'). If unknown reply N/A." if is_school
                       else "Approximate number of members/clients/visitors annually. If unknown reply N/A."),
        "Ratio":      "Staff-to-student or staff-to-client ratio (e.g. '1:8'). If unknown reply N/A.",
        "Fees":       "Annual fee or membership cost with currency symbol. SG international school fees are typically S$20,000–S$50,000/year. If unknown reply 'See website'.",
        "About":      f"2 sentences max (≤40 words) describing {name}'s core identity and mission.",
        "Philosophy": f"2 sentences max (≤40 words) on {name}'s teaching/coaching/wellness approach or ethos.",
        "Outcomes":   f"2 sentences max (≤40 words) on what {name} achieves for its students/members.",
        "Admissions": f"2 sentences max (≤40 words) on the enrolment/membership process at {name}.",
        "Tagline":    f"One compelling tagline (≤20 words) for {name}.",
    }

    ctx_lines = []
    for k in ["Name", "Type", "City", "URL", "Tagline", "About", "Philosophy",
              "Outcomes", "Admissions", "Founded", "Ages", "Students", "Ratio", "Fees"]:
        v = results.get(k, "")
        if v:
            ctx_lines.append(f"{k}: {v}")
    context = "\n".join(ctx_lines) or f"URL: {results.get('URL', '')}"

    def _needs(val):
        return not val or str(val).strip() in ("N/A", "See website", "Rolling admissions",
                                                "Year-round", "", "N/a")

    missing = {f: p for f, p in FILL_FIELDS.items() if _needs(results.get(f))}
    if not missing:
        return results

    print(f"  → OpenAI filling {len(missing)} main field(s): {', '.join(missing.keys())}")

    field_instructions = "\n".join(f'  "{k}": {v}' for k, v in missing.items())
    prompt = f"""You are a Singapore education and culture expert.
Use the known data AND your knowledge to fill missing fields accurately for {name}.

KNOWN DATA:
{context}

Fill ONLY these fields (reply with a JSON object, keys exactly as shown):
{field_instructions}

Rules:
- Use factual knowledge of this specific Singapore institution.
- Fees: use S$ and per year/term. SG international schools: S$20,000–S$50,000/year.
- Ages: always provide a range even for non-schools (e.g. 'Adults 18+', 'All ages').
- Keep paragraph fields SHORT: max 2 sentences, ≤40 words total.
- Do NOT add commentary, markdown fences, or extra keys.
- Reply with valid JSON only."""

    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 1500, "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            raw = data["choices"][0]["message"]["content"].strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw = re.sub(r"\s*```$", "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            for fkey, val in filled.items():
                if not val or str(val).strip() in ("N/A", "See website", ""):
                    continue
                if fkey in FILL_FIELDS and _needs(results.get(fkey)):
                    results[fkey] = str(val).strip()
                    filled_keys.append(fkey)

            if filled_keys:
                print(f"  ✓ OpenAI filled: {', '.join(filled_keys)}")
            else:
                print("  ✓ OpenAI: nothing new to fill.")
    except Exception as e:
        print(f"  ⚠ OpenAI main fill failed: {e}")

    return results


# ── WP BLOCKS GENERATOR ─

def generate_wp_blocks(r):
    school_name  = r.get("Name", "")
    description  = r.get("Tagline", "")
    age          = r.get("Ages", "")
    students     = r.get("Students", "")
    ratio        = r.get("Ratio", "")
    founded      = r.get("Founded", "")
    school_type  = r.get("Type", "")
    city         = r.get("City", "")
    annual_fee   = r.get("Fees", "")
    about        = r.get("About", "")
    philosophy   = r.get("Philosophy", "")
    outcomes     = r.get("Outcomes", "")
    quote        = r.get("Quote", "")
    website_url  = r.get("URL", "")
    admissions   = r.get("Admissions", "")
    images       = r.get("Images", [])
    perf         = r.get("Performance", {})
    enquiry_open = r.get("EnquiryOpen", "Year-round")
    app_deadline = r.get("AppDeadline", "Rolling admissions")

    website_display = website_url.replace("https://", "").replace("http://", "").rstrip("/")
    city_slug  = city.lower().replace(", ", "-").replace(" ", "-").replace(",", "")
    city_short = city.split(",")[0].strip()

    # Image slots — always have at least img0 for all 5
    imgs = [img for img in images if img]
    while len(imgs) < 6:
        imgs.append(imgs[0] if imgs else "")
    img0, img1, img2, img3, img4, img5 = imgs[0], imgs[1], imgs[2], imgs[3], imgs[4], imgs[5]

    coaching       = perf.get("Coaching Credentials", "")
    wellbeing      = perf.get("Student Wellbeing", "")
    academic       = perf.get("Academic Integration", "")
    competitive    = perf.get("Competitive Pathway", "")
    facilities     = perf.get("Facilities & Resources", "")
    accountability = perf.get("Ongoing Accountability", "")

    # Short keyword labels for WP
    kw_coaching       = PERF_SHORT["Coaching Credentials"]
    kw_wellbeing      = PERF_SHORT["Student Wellbeing"]
    kw_academic       = PERF_SHORT["Academic Integration"]
    kw_competitive    = PERF_SHORT["Competitive Pathway"]
    kw_facilities     = PERF_SHORT["Facilities & Resources"]
    kw_accountability = PERF_SHORT["Ongoing Accountability"]

    wp = f"""<!-- wp:image {{"id":440,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img0}" alt="{school_name}" class="wp-image-440"/></figure>
<!-- /wp:image -->

<!-- wp:paragraph -->
<p><a href="elite-home.html">Elite</a> &#8250; <a href="elite-city-{city_slug}.html">{city_short}</a> &#8250; <a href="elite-program-academic.html">Elite Academic Programs</a></p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Kidrovia Elite — Listed</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Elite Academic Programs</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":1}} -->
<h1 class="wp-block-heading"><em>{school_name}</em></h1>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{description}</p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{age}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{students}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{ratio}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{founded}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column {{"width":"66.66%"}} -->
<div class="wp-block-column" style="flex-basis:66.66%">
<!-- wp:heading --><h2 class="wp-block-heading">About <em>{school_name}</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{about}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column {{"width":"33.33%"}} -->
<div class="wp-block-column" style="flex-basis:33.33%">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Institution Details</h5><!-- /wp:heading -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Type</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">City</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{school_type}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{age}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{students}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{ratio}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{founded}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{city}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{annual_fee}</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Contact &amp; Enquiry</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading"><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display} &#8594;</a></h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">Quote</h5>
<!-- /wp:heading -->
<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Philosophy</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">How they <em>teach</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{philosophy}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Outcomes</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Where students <em>go</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{outcomes}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img1}" alt="{school_name} campus"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img2}" alt="{school_name} students"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img3}" alt="{school_name} activities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img4}" alt="{school_name} facilities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img5}" alt="{school_name} community"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":4}} -->
<h4 class="wp-block-heading">Our Assessment</h4>
<!-- /wp:heading -->
<!-- wp:heading -->
<h2 class="wp-block-heading"><strong>How {school_name} performs</strong></h2>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_coaching}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Coaching Credentials</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{coaching}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_wellbeing}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Student Wellbeing</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{wellbeing}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_academic}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Academic Integration</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{academic}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_competitive}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Competitive Pathway</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{competitive}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_facilities}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Facilities &amp; Resources</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{facilities}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_accountability}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Ongoing Accountability</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{accountability}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading --><h2 class="wp-block-heading">Admissions &amp;<br><em>How to Apply</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{admissions}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Enquiries Open</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{enquiry_open}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Application Deadline</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{app_deadline}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{annual_fee}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Location</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{city}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Website</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display}</a></p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->
"""
    return wp


# ── MAIN SCRAPER

async def extract_school_data(url):
    async with async_playwright() as p:
        import platform
        headless = os.environ.get(
            "HEADLESS",
            "true" if platform.system() == "Linux" else "false"
        ).lower() == "true"
        browser = await p.chromium.launch(headless=headless, slow_mo=0 if headless else 200)
        context = await browser.new_context(viewport={"width": 1280, "height": 800})
        page = await context.new_page()

        results = {
            "Name": "", "Tagline": "", "Founded": "", "City": "",
            "Ages": "", "Students": "", "Ratio": "", "Fees": "",
            "Type": "Independent School",
            "About": "", "Philosophy": "", "Outcomes": "",
            "Admissions": "", "Quote": "",
            "EnquiryOpen": "Year-round",
            "AppDeadline": "Rolling admissions",
            "Performance": {}, "URL": url, "Images": [],
        }

        global_text_pool = []

        # ── Determine correct name from KNOWN_NAMES_SG first ──
        domain = url.split("/")[2].replace("www.", "")
        known_name = next((v for k, v in KNOWN_NAMES_SG.items() if k in domain), None)

        FALLBACKS = {
            "Founded": "", "Ages": "", "Students": "", "Ratio": "N/A", "Fees": "See website",
        }

        fee_patterns = [
            r'S\$\s*([\d,]+)(?:\.\d{2})?\s*(?:per\s+(?:year|term|annum|month|semester))?',
            r'SGD\s*([\d,]+)(?:\.\d{2})?\s*(?:per\s+(?:year|term|annum))?',
            r'tuition\s*(?:fee)?s?\s*(?:of|is|are|:)?\s*S\$\s*([\d,]+)',
            r'annual\s*(?:tuition|fee)s?\s*(?:of|is|:)?\s*S\$\s*([\d,]+)',
            r'\$\s*([\d,]+)(?:\.\d{2})?\s*per\s*(?:year|term|annum|semester)',
        ]

        def extract_fee_validated_sg(text: str) -> str:
            for pat in fee_patterns:
                for m in re.finditer(pat, text, re.I):
                    try:
                        num = int(m.group(1).replace(',', ''))
                        if 1000 <= num <= 100000:
                            return m.group(0).strip()
                    except (IndexError, ValueError):
                        pass
            return ""

        ratio_patterns = [
            r'(\d+\s*:\s*\d+)\s*(?:student[- ]to[- ](?:faculty|teacher)|teacher[- ]to[- ]student|ratio)',
            r'ratio\s*(?:of\s*)?(\d+\s*:\s*\d+)',
            r'(\d+\s*:\s*\d+)\s*ratio',
            r'1\s*(?:teacher|member of staff|coach)\s+(?:to|per|:)\s+(\d+)\s*(?:students?|pupils?|learners?)',
            r'(\d+)\s*(?:students?|pupils?|learners?)\s+(?:per|to)\s+(?:every\s+)?(?:1\s+)?(?:teacher|tutor|coach)',
        ]

        age_patterns = [
            r'students?\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'children\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'ages?\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'from\s+age\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'members?\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'(?:dancers?|singers?|musicians?|athletes?)\s+ages?\s+(\d+)[–\-](\d+)',
            r'(\d+)\s*(?:to|[\u2013\-])\s*(\d+)\s*years?\s*(?:old)',
        ]

        student_patterns = [
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:pupils?|students?|boys|girls|young people|young women|learners?|scholars?)',
            r'(?:approximately|around|about|some|over|more than)\s+(\d[\d,]+)\s*(?:pupils?|students?|girls?|learners?)',
            r'school\s+of\s+(?:approximately|around|about)?\s*(\d[\d,]+)',
            r'enroll(?:ment|ment|ing)\s+(?:of\s+)?(?:approximately|about)?\s*(\d[\d,]+)',
            r'community\s+of\s+(?:over\s+)?(\d[\d,]+)\s*(?:students?|learners?|members?)',
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:student|pupil|learner|child|member)',
        ]

        print(f"Connecting to: {url}...")
        loaded = await smart_goto(page, url)
        if not loaded:
            print(f"  ✗ Could not load {url}, skipping.")
            await browser.close()
            return None

        try:
            await handle_cookies_automatically(page)

            og_site  = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:site_name\"]'); return m ? m.content : ''; }")
            og_title = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:title\"]'); return m ? m.content : ''; }")
            raw_title = await page.title()

            REJECT_NAMES = {"home","welcome","homepage","index","main","","welcome to","about","contact"}
            REJECT_STARTS = ("welcome to ", "home -", "home |",
                             "best child", "top fitness", "international school in",
                             "british international school in")
            REJECT_PHRASES = [
                "personalization","advertising effectiveness",
                "tuition centre & enrichment classes",
                "top fitness & wellness professionals",
                "best child & adult psychologist",
                "pixelcommerce", "wix.com", "shopify",
            ]

            def clean_name(candidate: str) -> str:
                parts = re.split(r'\s*[|—\-]\s*', candidate)
                parts = [p.strip() for p in parts if p.strip()]
                GENERIC = {"home","welcome","index","main","page","site","portal","homepage","about",""}
                n = parts[0] if parts else candidate.strip()
                if n.lower() in GENERIC and len(parts) > 1:
                    n = parts[-1].strip()
                for prefix in ("welcome to ",):
                    if n.lower().startswith(prefix):
                        n = n[len(prefix):].strip()
                return n

            # Always prefer known name — avoids "PixelCommerce" type errors
            if known_name:
                results["Name"] = known_name
            else:
                for candidate in [og_site, og_title, raw_title]:
                    name = clean_name(candidate)
                    if (name
                            and name.lower() not in REJECT_NAMES
                            and not any(name.lower().startswith(p) for p in REJECT_STARTS)
                            and not any(bad in name.lower() for bad in REJECT_PHRASES)
                            and len(name.split()) <= 8):
                        results["Name"] = name
                        break
                if not results["Name"]:
                    results["Name"] = clean_name(raw_title)

            body_text = await page.inner_text("body")

            if len(body_text.strip()) < 300:
                print("  ⚠ Page body too short — scrolling to trigger JS render…")
                try:
                    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await asyncio.sleep(3)
                    await page.evaluate("window.scrollTo(0, 0)")
                    await asyncio.sleep(2)
                    body_text = await page.inner_text("body")
                except Exception:
                    pass

            results["City"] = detect_city(url, body_text)

            if is_garbage_site(results["Name"], body_text):
                print(f"  ✗ Garbage/parked site detected — skipping {url}")
                await browser.close()
                return None

            def extract(pattern):
                m = re.search(pattern, body_text, re.IGNORECASE)
                return m.group(1).strip() if m else ""

            current_year = datetime.now().year

            def extract_founded_from_text(text, existing_candidates=None):
                STRONG_KWS = ["founded in", "established in", "was founded", "founding year",
                               "founded by", "charter", "incorporated in", "opened in",
                               "opened its doors", "since", "est."]
                WEAK_KWS = ["founded", "established", "foundation", "history", "began"]
                SKIP_WORDS = ["copyright", "©", "reserved", "updated", "class of", "alumni"]
                FOREIGN_CTX = ["in the uk", "in england", "in derbyshire", "in scotland",
                                "in wales", "in ireland", "parent school", "sister school"]
                cands = existing_candidates if existing_candidates is not None else []
                clean = re.sub(r'\s+', ' ', text.lower())
                clean = re.sub(r'(copyright|©).*?\d{4}', '', clean)
                for sent in re.split(r'[.!?\n]', clean):
                    if any(w in sent for w in SKIP_WORDS):
                        continue
                    has_strong = any(k in sent for k in STRONG_KWS)
                    has_weak = any(k in sent for k in WEAK_KWS)
                    if not (has_strong or has_weak):
                        continue
                    is_foreign = any(ctx in sent for ctx in FOREIGN_CTX)
                    for y in re.findall(r'\b(1[6-9]\d{2}|20[012]\d)\b', sent):
                        year = int(y)
                        if not (1600 <= year <= current_year - 1):
                            continue
                        score = 3 if has_strong else 1
                        if year < 1900:
                            score -= 4
                        if is_foreign:
                            score -= 3
                        if year >= current_year - 2:
                            score -= 3
                        cands.append((score, year))
                return cands

            founded_candidates = extract_founded_from_text(body_text)
            if founded_candidates:
                founded_candidates.sort(key=lambda x: (-x[0], x[1]))
                results["Founded"] = str(founded_candidates[0][1])

            # Ages
            age_candidates = []
            for pat in age_patterns:
                for m in re.finditer(pat, body_text, re.I):
                    try:
                        g = m.groups()
                        if len(g) >= 2 and g[0] and g[1]:
                            lo, hi = int(g[0]), int(g[1])
                            if lo < hi and lo >= 2 and hi <= 25:
                                age_candidates.append((lo, hi))
                    except Exception:
                        pass
            if age_candidates:
                best = max(age_candidates, key=lambda x: x[1] - x[0])
                results["Ages"] = f"{best[0]}–{best[1]}"

            if not results["Ages"]:
                AGE_MAP = {
                    "nursery": (2, 4), "kindergarten": (4, 6), "k1": (4, 5), "k2": (5, 6),
                    "pre-school": (2, 6), "preschool": (2, 6),
                    "primary 1": (7, 8), "primary 6": (12, 13), "p1": (7, 8), "p6": (12, 13),
                    "psle": (12, 13), "secondary 1": (13, 14), "secondary 4": (16, 17),
                    "sec 1": (13, 14), "sec 4": (16, 17), "secondary 5": (17, 18),
                    "o-level": (16, 17), "a-level": (17, 19), "jc1": (17, 18), "jc2": (18, 19),
                    "junior college": (17, 19), "integrated programme": (13, 19),
                    "ib diploma": (16, 18), "igcse": (14, 16), "middle years": (11, 16),
                    "primary years": (3, 12), "grade 1": (6, 7), "grade 12": (17, 18),
                    "year 7": (11, 12), "year 13": (17, 18), "elementary": (5, 11),
                    "middle school": (11, 14), "high school": (14, 18), "upper school": (14, 18),
                }
                tl = body_text.lower()
                fy = [v for k, v in AGE_MAP.items() if k in tl]
                results["Ages"] = f"{min(x[0] for x in fy)}–{max(x[1] for x in fy)}" if fy else ""

            # Students
            for pat in student_patterns:
                for m in re.finditer(pat, body_text, re.I):
                    num_str = re.sub(r'[\s,]', '', m.group(1))
                    try:
                        num = int(num_str)
                        if not (50 <= num <= 10000):
                            continue
                        if 2020 <= num <= 2030:
                            continue
                        ctx = body_text[max(0, m.start()-8):m.end()+8]
                        if '%' in ctx or '.' in m.group(1):
                            continue
                        results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                        break
                    except ValueError:
                        pass
                if results["Students"]:
                    break

            # Ratio
            for pat in ratio_patterns:
                m = re.search(pat, body_text, re.I)
                if m:
                    results["Ratio"] = m.group(1).replace(" ", "")
                    break

            results["Fees"] = extract_fee_validated_sg(body_text)

            # Type detection
            tl = body_text.lower()
            url_l = url.lower()
            if "tts.edu.sg" in url_l:
                results["Type"] = "British International School"
            elif "sas.edu.sg" in url_l:
                results["Type"] = "American International School"
            elif "ofs.edu.sg" in url_l or "cis.edu.sg" in url_l:
                results["Type"] = "IB World Day School"
            elif "sportsschool.edu.sg" in url_l:
                results["Type"] = "Specialised Sports School"
            elif "sota.edu.sg" in url_l:
                results["Type"] = "Specialised Arts School"
            elif "lasalle.edu.sg" in url_l:
                results["Type"] = "Arts College & Conservatoire"
            elif "srmc.edu.sg" in url_l:
                results["Type"] = "Arts & Music College"
            elif "sso.org.sg" in url_l or "snyo" in url_l:
                results["Type"] = "National Youth Orchestra"
            elif "sswimclub.org.sg" in url_l:
                results["Type"] = "Swimming Club & Academy"
            elif "tanglinclub.org" in url_l:
                results["Type"] = "Private Members Club"
            elif "scc.org.sg" in url_l:
                results["Type"] = "Sports & Social Club"
            elif "prep-zone.sg" in url_l:
                results["Type"] = "Academic Tutoring & Test Prep"
            elif "thelearninglab.com.sg" in url_l:
                results["Type"] = "Tuition & Enrichment Centre"
            elif "crimsoneducation.org" in url_l:
                results["Type"] = "College Admissions Consultancy"
            elif "brahmcentre.com" in url_l:
                results["Type"] = "Mindfulness & Wellness Centre"
            elif "centreformindfulness.sg" in url_l:
                results["Type"] = "Mindfulness & Wellness Centre"
            elif "redefinewellness.asia" in url_l:
                results["Type"] = "Corporate Wellness & Mindfulness"
            elif "corecollective.sg" in url_l:
                results["Type"] = "Fitness & Wellness Hub"
            elif "psychconnect.sg" in url_l:
                results["Type"] = "Psychology & Counselling Centre"
            elif "ib world" in tl or "international baccalaureate" in tl:
                results["Type"] = "IB World School"
            elif "british curriculum" in tl or "igcse" in tl or "a-level" in tl:
                results["Type"] = "British Curriculum International School"
            elif "all-girls" in tl or "girls' school" in tl:
                results["Type"] = "Independent All-Girls School"
            elif "tuition centre" in tl or "tuition center" in tl or "learning centre" in tl:
                results["Type"] = "Tuition & Enrichment Centre"
            elif "conservatoire" in tl or "conservatory" in tl:
                results["Type"] = "Arts Conservatory"
            elif "swim" in tl and ("club" in tl or "academy" in tl):
                results["Type"] = "Swimming Club & Academy"
            elif "mindfulness" in tl or "meditation" in tl:
                results["Type"] = "Mindfulness & Wellness Centre"
            elif "wellness" in tl or "wellbeing" in tl:
                results["Type"] = "Wellness Centre"
            elif "psychology" in tl or "counselling" in tl:
                results["Type"] = "Psychology & Counselling Centre"
            elif "boarding" in tl and "day" in tl:
                results["Type"] = "Independent Boarding & Day School"
            else:
                results["Type"] = "Independent School"

            soup_home = BeautifulSoup(await page.content(), "html.parser")
            for attr in [{"name": "description"}, {"property": "og:description"}]:
                tag = soup_home.find("meta", attr)
                if tag and tag.get("content"):
                    results["Tagline"] = tag["content"][:200]
                    break
            if not results["Tagline"]:
                results["Tagline"] = extract(r"([A-Z][^.]{30,150}\.)")

            results["Quote"] = scrape_quote_from_soup(soup_home)

            image_set = set()
            await harvest_images(page, url, image_set)
            results["Images"] = [img for img in image_set if img][:10]

        except Exception as e:
            print(f"Home page error: {e}")
            await browser.close()
            return None

        # ── SUBPAGE CRAWL 
        soup = BeautifulSoup(await page.content(), "html.parser")
        links = soup.find_all("a", href=True)
        queue = []
        seen_urls = {url.rstrip("/")}

        targets = {
            "about": ["about", "history", "welcome", "mission", "who-we-are",
                      "our-story", "overview", "vision", "values", "ethos"],
            "outcomes": ["results", "destination", "university", "leavers", "beyond",
                         "sixth-form", "exam", "college", "achievements", "academic-results"],
            "fees": ["fees", "tuition", "financial", "cost", "charges",
                     "fee-structure", "school-fees", "annual-fees", "pricing"],
            "admission": ["admissions", "apply", "entry", "register", "enroll",
                          "how-to-apply", "application", "join-us", "registration", "enrolment"],
            "facilities": ["facilities", "campus", "co-curriculum", "sport", "arts",
                           "library", "infrastructure", "resources", "activities",
                           "co-curricular", "extracurricular"],
        }

        for link in links:
            href, text = link["href"], link.get_text().lower().strip()
            full_url = urljoin(url, href).split("#")[0].rstrip("/")
            if full_url not in seen_urls and url.split("/")[2] in full_url:
                if any(k in text or k in href.lower() for cat in targets.values() for k in cat):
                    queue.append((full_url, text))
                    seen_urls.add(full_url)

        used_sentences: set = set()

        def _register(text: str):
            if not text:
                return
            for s in re.split(r'(?<=[.!?])\s+', text.strip()):
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and len(key) > 20:
                    used_sentences.add(key)

        def dedup_text(text: str) -> str:
            if not text:
                return text
            parts = re.split(r'(?<=[.!?])\s+', text.strip())
            fresh = []
            for s in parts:
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and key not in used_sentences and not is_junk(s):
                    fresh.append(s.strip())
                    used_sentences.add(key)
            return " ".join(fresh)

        def assign_field(field_name: str, text: str, max_sent: int = 2):
            if results.get(field_name):
                return
            cleaned = dedup_text(clean_to_sentences(text, max_sent))
            if cleaned:
                results[field_name] = cleaned

        for _field in ["About", "Tagline", "Philosophy", "Outcomes", "Admissions", "Quote"]:
            _register(results.get(_field, ""))

        used_paras_global: set = set()

        for target_url, link_text in queue[:15]:
            try:
                await page.goto(target_url, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(2)
                main = page.locator("main").first
                content_scope = main if await main.is_visible() else page
                paragraphs = await content_scope.locator("p").all_text_contents()
                clean_paras = [
                    t.strip() for t in paragraphs
                    if len(t.strip()) >= 80
                    and not any(x in t.lower() for x in ["cookie", "privacy", "menu", "navigation", "footer"])
                    and not is_junk(t)
                ]
                global_text_pool.extend(clean_paras)
                full_text = " ".join(clean_paras)
                page_text = await page.inner_text("body")
                sub_soup = BeautifulSoup(await page.content(), "html.parser")

                await harvest_images(page, url, image_set)
                results["Images"] = [img for img in image_set if img][:10]

                if any(k in link_text or k in target_url for k in targets["about"]):
                    clean_about = [p for p in clean_paras if not is_junk(p)
                                   and not p.lower().startswith(("please fill", "please note",
                                   "the admissions team", "fill the", "click here", "select"))]
                    assign_field("About", " ".join(clean_about), 2)
                    teach_kws = ["curriculum", "teaching", "learning", "approach",
                                 "education", "ethos", "method", "ib", "igcse", "philosophy"]
                    teach_para = [p for p in clean_paras if any(k in p.lower() for k in teach_kws)]
                    assign_field("Philosophy", " ".join(teach_para), 2)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)
                    if not results["Founded"]:
                        sub_cands = extract_founded_from_text(page_text)
                        if sub_cands:
                            sub_cands.sort(key=lambda x: (-x[0], x[1]))
                            results["Founded"] = str(sub_cands[0][1])

                if not results["Students"]:
                    for pat in student_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            num_str = m.group(1).replace(",", "")
                            try:
                                num = int(num_str)
                                if 50 <= num <= 10000:
                                    results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                                    break
                            except ValueError:
                                pass

                if not results["Ratio"]:
                    for pat in ratio_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            results["Ratio"] = m.group(1).replace(" ", "")
                            break

                if any(k in link_text or k in target_url for k in targets["outcomes"]):
                    assign_field("Outcomes", full_text, 2)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)

                if any(k in link_text or k in target_url for k in targets["fees"]):
                    fee_val = extract_fee_validated_sg(page_text)
                    if fee_val and not results["Fees"]:
                        results["Fees"] = fee_val

                if any(k in link_text or k in target_url for k in targets["admission"]):
                    adm_kws = ["apply", "application", "admission", "register",
                               "assessment", "entry", "enroll", "enrolment", "registration"]
                    filtered = [p for p in clean_paras if any(k in p.lower() for k in adm_kws)]
                    assign_field("Admissions", " ".join(filtered), 2)
                    dl = re.search(
                        r'(?:deadline|closes?|closing date)\s*[:\-]?\s*([^\n.]{10,60})',
                        page_text, re.I
                    )
                    if dl and results["AppDeadline"].startswith("Rolling"):
                        val = dl.group(1).strip()[:80]
                        has_month = re.search(
                            r'\b(january|february|march|april|may|june|july|august|'
                            r'september|october|november|december|jan|feb|mar|apr|'
                            r'jun|jul|aug|sep|oct|nov|dec)\b', val, re.I)
                        has_date = re.search(r'\b\d{1,2}[\/\-]\d{1,2}', val)
                        if (has_month or has_date) and not is_junk(val):
                            results["AppDeadline"] = val
                    if not results["Ages"]:
                        for pat in age_patterns:
                            m = re.search(pat, page_text, re.I)
                            if m:
                                try:
                                    g = m.groups()
                                    if len(g) >= 2 and g[0] and g[1]:
                                        lo, hi = int(g[0]), int(g[1])
                                        if lo < hi and lo >= 2 and hi <= 25:
                                            results["Ages"] = f"{lo}–{hi}"
                                except Exception:
                                    pass
                                if results["Ages"]:
                                    break

                if any(k in link_text or k in target_url for k in targets["facilities"]):
                    perf_kws = {
                        "Coaching Credentials": ["qualified teacher", "experienced staff",
                                                  "certified", "credentials", "instructor",
                                                  "faculty", "expertise"],
                        "Student Wellbeing": ["wellbeing", "pastoral", "counselor",
                                               "mental health", "welfare", "house system",
                                               "nurturing", "support programme"],
                        "Academic Integration": ["curriculum", "igcse", "a-level", "ib",
                                                  "programme of study", "academic programme",
                                                  "learning outcomes", "subject"],
                        "Competitive Pathway": ["university", "destination", "oxford",
                                                 "cambridge", "results", "examination",
                                                 "achievement", "ranking"],
                        "Facilities & Resources": ["facilities", "campus", "library",
                                                    "laboratory", "studio", "theatre",
                                                    "swimming pool", "gymnasium", "court"],
                        "Ongoing Accountability": ["progress", "tracking", "monitoring",
                                                    "feedback", "report", "review",
                                                    "assessment cycle", "parent portal"],
                    }
                    for key, kws in perf_kws.items():
                        if not results["Performance"].get(key):
                            matched = [
                                p for p in clean_paras
                                if any(k in p.lower() for k in kws)
                                and p not in used_paras_global
                                and not is_junk(p)
                            ]
                            if matched:
                                chosen = matched[:1]
                                val = dedup_text(clean_to_sentences(" ".join(chosen), 2))
                                if val:
                                    results["Performance"][key] = val
                                    used_paras_global.update(chosen)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)

            except Exception as e:
                print(f"  Skipping {target_url}: {e}")
                continue

        results["Images"] = list(image_set)[:10]
        for field, fallback in FALLBACKS.items():
            if not results.get(field):
                results[field] = fallback
        if not results["City"]:
            results["City"] = "Singapore"

        if not results["Quote"]:
            print(f"  No quote found — generating AI quote…")
            results["Quote"] = generate_ai_quote(
                results["Name"], results["Type"], results["City"], results["About"]
            )

        await browser.close()

        # ── STEP 1: Fill main fields 
        print(f"\n  Running OpenAI main field gap-fill…")
        results = openai_fill_missing_fields(results)

        # ── STEP 2: Force-fill ALL performance fields (no N/A allowed) 
        print(f"\n  Running AI force-fill for performance fields…")
        results = openai_force_fill_performance(results)

        # ── POST SANITIZATION 
        for field in ["Name", "Tagline", "Founded", "City", "Ages", "Students",
                      "Ratio", "Fees", "Type", "About", "Philosophy", "Outcomes",
                      "Admissions", "Quote"]:
            if results.get(field):
                results[field] = re.sub(r'\s+', ' ', str(results[field])).strip()
        for field in ["About", "Philosophy", "Outcomes", "Admissions"]:
            if results.get(field) and len(results[field].split()) > 50:
                results[field] = clean_to_sentences(results[field], 2)
        PERF_METRICS = ["Coaching Credentials", "Student Wellbeing", "Academic Integration",
                        "Competitive Pathway", "Facilities & Resources", "Ongoing Accountability"]
        for metric in PERF_METRICS:
            val = results["Performance"].get(metric, "")
            if val and len(val.split()) > 50:
                results["Performance"][metric] = clean_to_sentences(val, 2)

        # ── PRINT SUMMARY
        SEP = "—" * 55
        city_short = results["City"].split(",")[0]
        print(f"\n{SEP}")
        print(f"Elite › {city_short} › {results['Name']}")
        print(SEP)
        print(f"\n  Type       : {results['Type']}")
        print(f"  Ages       : {results['Ages'] or 'N/A'}")
        print(f"  Students   : {results['Students'] or 'N/A'}")
        print(f"  Ratio      : {results['Ratio'] or 'N/A'}")
        print(f"  Founded    : {results['Founded'] or 'N/A'}")
        print(f"  City       : {results['City']}")
        print(f"  Annual Fee : {results['Fees']}")
        print(f"  Images     : {len(results['Images'])} found")
        for i, img in enumerate(results["Images"], 1):
            print(f"    {i}. {img}")
        print(f"\n  About      : {results['About']}")
        print(f"\n  Quote      : \"{results['Quote']}\"")
        print(f"\n  Philosophy : {results['Philosophy']}")
        print(f"\n  Outcomes   : {results['Outcomes']}")
        print(f"\n  Admissions : {results['Admissions']}")
        print(f"\nPerformance Metrics:")
        for k, v in results["Performance"].items():
            kw = PERF_SHORT.get(k, k)
            print(f"\n  [{kw}] {k}:")
            print(f"  {v or ' STILL MISSING'}")
        print(f"\n  Website : {results['URL']}")
        print(SEP)

        # ── SAVE 
        wp_code = generate_wp_blocks(results)
        school_slug = re.sub(r"[^a-z0-9]+", "_", results["Name"].lower())[:40].strip("_")
        out_wp   = f"{school_slug}_wp_blocks.txt"
        out_json = f"{school_slug}_data.json"

        with open(out_wp, "w", encoding="utf-8") as f:
            f.write(wp_code)

        save_data = {k: v for k, v in results.items() if k != "Images"}
        save_data["ImageCount"] = len(results.get("Images", []))
        save_data["ImageURLs"]  = results.get("Images", [])
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(save_data, f, ensure_ascii=False, indent=2)

        print(f"\n  WP blocks  → {out_wp}")
        print(f"  JSON data  → {out_json}")
        return results


# ── URLS 

school_urls = [
    "https://tts.edu.sg",
    "https://ofs.edu.sg",
    "https://sas.edu.sg",
    "https://cis.edu.sg",
    "https://sportsschool.edu.sg",
    "https://sswimclub.org.sg/",
    "https://www.tanglinclub.org/",
    "https://scc.org.sg/youth-development/",
    "https://prep-zone.sg/",
    "https://thelearninglab.com.sg",
    "https://www.crimsoneducation.org/sg",
    "https://lasalle.edu.sg",
    "https://www.sso.org.sg/snyo",
    "https://brahmcentre.com",
    "https://centreformindfulness.sg",
    "https://redefinewellness.asia",
    "https://corecollective.sg",
    "https://psychconnect.sg",
    "https://srmc.edu.sg/",
    "https://www.sota.edu.sg/",
]


async def run_multiple_schools():
    all_results = []
    for url in school_urls:
        print(f"\n{'=' * 60}")
        print(f"  Scraping: {url}")
        print(f"{'=' * 60}")
        try:
            result = await extract_school_data(url)
            if result:
                all_results.append(result)
            await asyncio.sleep(3)
        except Exception as e:
            print(f"  Failed: {url} → {e}")

    master_file = "all_schools_data_singapore.json"
    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n{'=' * 60}")
    print(f"All done! {len(all_results)} institutions scraped.")
    print(f"Master JSON → {master_file}")
    print(f"{'=' * 60}")
    return all_results


if __name__ == "__main__":
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future   = pool.submit(asyncio.run, run_multiple_schools())
            all_data = future.result()
    except RuntimeError:
        asyncio.run(run_multiple_schools())


  Scraping: https://tts.edu.sg
Connecting to: https://tts.edu.sg...

  Running OpenAI main field gap-fill…
  → OpenAI filling 2 main field(s): Ratio, Fees
  ✓ OpenAI filled: Ratio, Fees

  Running AI force-fill for performance fields…
  → AI force-filling 6 performance field(s): Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability
  ✓ AI filled performance: Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability

———————————————————————————————————————————————————————
Elite › Singapore › Tanglin Trust School
———————————————————————————————————————————————————————

  Type       : British International School
  Ages       : 3–7
  Students   : 178
  Ratio      : 1:10
  Founded    : 1925
  City       : Singapore
  Annual Fee : S$30,000–S$45,000 per year
  Images     : 10 found
    1. https://resources.finalsite.net/images/f_auto,q_auto/v1